# 🤖 Machine Learning Demo Apps
## Time Series Forecasting & Customer Segmentation

Notebook ini berisi dua aplikasi Streamlit interaktif untuk kelas Machine Learning:

| App | Deskripsi | Port |
|-----|-----------|------|
| 🔮 **Time Series Forecasting** | Peramalan deret waktu dengan Supervised Learning | 8501 |
| 👥 **Customer Segmentation** | Segmentasi pelanggan dengan KMeans Clustering | 8502 |

### Langkah Penggunaan:
1. Jalankan **Cell 1** — install dependencies
2. Jalankan **Cell 2** — masukkan ngrok authtoken Anda
3. Jalankan **Cell 3** — tulis file `app_timeseries.py`
4. Jalankan **Cell 4** — tulis file `app_segmentasi.py`
5. Jalankan **Cell 5** — untuk membuka App 1 (Time Series)
6. Jalankan **Cell 6** — untuk membuka App 2 (Segmentasi)
7. *(Opsional)* Jalankan **Cell 7** — untuk menghentikan semua tunnel

---
## ⚙️ CELL 1 — Install Dependencies

In [ ]:
!pip install streamlit pyngrok plotly scikit-learn pandas numpy yfinance

---
## 🔑 CELL 2 — Setup Ngrok Authtoken

> **Dapatkan token gratis di:** https://dashboard.ngrok.com/get-started/your-authtoken  
> Ganti `YOUR_NGROK_AUTHTOKEN_HERE` dengan token Anda.

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")

---
## 🔮 CELL 3 — Tulis File `app_timeseries.py`

In [ ]:
%%writefile app_timeseries.py
# ============================================================
# APP 1: TIME SERIES FORECASTING DEMO
# Supervised Learning untuk peramalan deret waktu
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import yfinance as yf
import io
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="🔮 Time Series Forecasting Demo",
    page_icon="🔮",
    layout="wide"
)

st.title("🔮 Time Series Forecasting")
st.markdown(
    "Demo interaktif peramalan deret waktu menggunakan "
    "Supervised Learning. Ubah parameter di sidebar untuk "
    "melihat bagaimana model bereaksi."
)

# ── Fungsi: buat data demo sintetis ─────────────────────────
def buat_data_demo():
    """Generate 36 bulan data penjualan sintetis yang reproducible."""
    rng = np.random.RandomState(42)
    n = 36
    tanggal = pd.date_range(start="2022-01", periods=n, freq="MS")
    tren = np.linspace(100, 250, n)
    # Seasonal: naik bulan 11-12, turun bulan 1-2
    seasonal = np.array([
        -20 if m in [1, 2] else (40 if m in [11, 12] else 0)
        for m in tanggal.month
    ])
    noise = rng.normal(0, 10, n)
    nilai = tren + seasonal + noise
    # Satu anomali kecil di sekitar bulan ke-18
    nilai[17] += 60
    df = pd.DataFrame({"tanggal": tanggal, "nilai": np.round(nilai, 2)})
    return df

# ── Fungsi: buat lag features ────────────────────────────────
def buat_lag_features(df, kolom_nilai, n_lag):
    """Tambahkan kolom lag_1 sampai lag_n ke dataframe."""
    df_lag = df.copy()
    for i in range(1, n_lag + 1):
        df_lag[f"lag_{i}"] = df_lag[kolom_nilai].shift(i)
    return df_lag.dropna().reset_index(drop=True)

# ── Fungsi: pilih model ──────────────────────────────────────
def pilih_model(nama_model):
    if nama_model == "Linear Regression":
        return LinearRegression()
    elif nama_model == "Ridge Regression":
        return Ridge(alpha=1.0)
    else:
        return DecisionTreeRegressor(random_state=42)

# ── Fungsi: hitung metrik ────────────────────────────────────
def hitung_metrik(y_aktual, y_prediksi):
    mae = mean_absolute_error(y_aktual, y_prediksi)
    rmse = np.sqrt(mean_squared_error(y_aktual, y_prediksi))
    r2 = r2_score(y_aktual, y_prediksi)
    return mae, rmse, r2

# ── Fungsi: hitung baseline naive ───────────────────────────
def hitung_baseline_naive(y_aktual, y_train_last):
    """Baseline: prediksi = nilai sebelumnya (naive forecast)."""
    y_naive = np.concatenate([[y_train_last], y_aktual[:-1]])
    mae = mean_absolute_error(y_aktual, y_naive)
    rmse = np.sqrt(mean_squared_error(y_aktual, y_naive))
    r2 = r2_score(y_aktual, y_naive)
    return mae, rmse, r2

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    # Pilihan sumber data
    sumber_data = st.radio(
        "Pilih Sumber Data",
        options=[
            "📊 Data Demo (Penjualan Sintetis)",
            "📈 Data Saham Live (via yfinance)",
            "📁 Upload CSV Sendiri"
        ]
    )

    df_data = None
    nama_dataset = "Data Demo"
    interval_data = "bulanan"

    # ── Opsi 1: Data Demo ────────────────────────────────────
    if sumber_data == "📊 Data Demo (Penjualan Sintetis)":
        df_data = buat_data_demo()
        nama_dataset = "Penjualan Sintetis"
        interval_data = "bulanan"

    # ── Opsi 2: Data Saham ───────────────────────────────────
    elif sumber_data == "📈 Data Saham Live (via yfinance)":
        ticker_input = st.text_input(
            "Kode Saham",
            value="BBCA.JK",
            placeholder="Contoh: BBCA.JK, TLKM.JK, AAPL, TSLA"
        )
        periode_map = {"1 Tahun": "1y", "2 Tahun": "2y", "3 Tahun": "3y"}
        periode_label = st.selectbox("Periode Data", list(periode_map.keys()))

        interval_map = {"Harian": "1d", "Mingguan": "1wk", "Bulanan": "1mo"}
        interval_label = st.selectbox("Interval", list(interval_map.keys()))

        if st.button("🔍 Ambil Data"):
            try:
                raw = yf.download(
                    ticker_input.strip(),
                    period=periode_map[periode_label],
                    interval=interval_map[interval_label],
                    progress=False
                )
                if raw.empty or "Close" not in raw.columns:
                    raise ValueError(f"Ticker '{ticker_input}' tidak ditemukan atau tidak ada data.")
                raw = raw.reset_index()
                kolom_tanggal = "Datetime" if "Datetime" in raw.columns else "Date"
                df_saham = raw[[kolom_tanggal, "Close"]].copy()
                # Flatten MultiIndex kolom jika ada
                df_saham.columns = ["tanggal", "nilai"]
                df_saham["nilai"] = pd.to_numeric(df_saham["nilai"], errors="coerce")
                df_saham = df_saham.dropna().reset_index(drop=True)
                st.session_state["df_saham"] = df_saham
                st.session_state["nama_saham"] = ticker_input.strip()
                st.session_state["interval_saham"] = interval_label
                st.info(f"✅ Berhasil mengambil {len(df_saham)} baris data untuk **{ticker_input.strip()}**")
            except Exception as e:
                st.error(f"❌ Gagal mengambil data: {e}\n\nMenggunakan data demo sebagai pengganti.")
                st.session_state["df_saham"] = buat_data_demo()
                st.session_state["nama_saham"] = "Demo (fallback)"
                st.session_state["interval_saham"] = "Bulanan"

        if "df_saham" in st.session_state:
            df_data = st.session_state["df_saham"]
            nama_dataset = f"Saham {st.session_state.get('nama_saham', ticker_input)}"
            interval_data = st.session_state.get("interval_saham", "Bulanan").lower()
        else:
            df_data = buat_data_demo()
            nama_dataset = "Data Demo (belum fetch)"
            interval_data = "bulanan"

        st.warning(
            "⚠️ Data saham bersifat sangat fluktuatif (noisy). Model sederhana "
            "seperti Linear Regression mungkin menghasilkan akurasi rendah — "
            "hal ini wajar dan justru menjadi pembelajaran penting: tidak semua "
            "model cocok untuk semua jenis data."
        )

    # ── Opsi 3: Upload CSV ───────────────────────────────────
    else:
        st.markdown(
            "**Format CSV:** 2 kolom wajib — kolom tanggal "
            "(format `YYYY-MM` atau `YYYY-MM-DD`) dan kolom nilai numerik."
        )
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])
        if file_upload is not None:
            try:
                df_upload = pd.read_csv(file_upload)
                st.write("**Preview 5 baris pertama:**")
                st.dataframe(df_upload.head())
                kolom_tersedia = list(df_upload.columns)
                kolom_tgl = st.selectbox("Pilih kolom tanggal", kolom_tersedia, index=0)
                kolom_val = st.selectbox(
                    "Pilih kolom nilai",
                    kolom_tersedia,
                    index=min(1, len(kolom_tersedia) - 1)
                )
                try:
                    df_csv = df_upload[[kolom_tgl, kolom_val]].copy()
                    df_csv.columns = ["tanggal", "nilai"]
                    df_csv["tanggal"] = pd.to_datetime(df_csv["tanggal"])
                    df_csv["nilai"] = pd.to_numeric(df_csv["nilai"], errors="coerce")
                    df_csv = df_csv.dropna().sort_values("tanggal").reset_index(drop=True)
                    if len(df_csv) < 10:
                        raise ValueError("Data terlalu sedikit (minimal 10 baris).")
                    df_data = df_csv
                    nama_dataset = file_upload.name
                    interval_data = "bulanan"
                except Exception as e:
                    st.error(f"❌ Gagal memproses CSV: {e}\n\nMenggunakan data demo.")
                    df_data = buat_data_demo()
                    nama_dataset = "Data Demo (fallback)"
                    interval_data = "bulanan"
            except Exception as e:
                st.error(f"❌ Gagal membaca file: {e}")
                df_data = buat_data_demo()
                nama_dataset = "Data Demo (fallback)"
                interval_data = "bulanan"
        else:
            df_data = buat_data_demo()
            nama_dataset = "Data Demo (menunggu upload)"
            interval_data = "bulanan"

    st.divider()

    # Pilihan model
    model_dipilih = st.selectbox(
        "Pilih Model",
        ["Linear Regression", "Ridge Regression", "Decision Tree Regressor"]
    )

    # Jumlah periode forecast
    n_forecast = st.slider("Jumlah Bulan/Periode Ramalan ke Depan", 1, 12, 6)

    # Jumlah lag features
    n_lag = st.slider("Jumlah Lag Features", 1, 12, 3)

    # Penjelasan lag features
    with st.expander("ℹ️ Apa itu Lag Features?"):
        st.markdown("""
**Lag features** adalah nilai dari waktu sebelumnya yang digunakan sebagai fitur input model.

**Contoh dengan lag=3:**
- Fitur: `nilai_t-1`, `nilai_t-2`, `nilai_t-3`
- Target: `nilai_t`

Semakin banyak lag → model "mengingat" lebih jauh ke masa lalu.  
Terlalu sedikit lag → model kurang informasi.  
Terlalu banyak lag → model bisa overfit pada data kecil.

*(Lag features = cara mengubah time series menjadi supervised learning problem.)*
        """)

# ── Pastikan df_data tidak None ──────────────────────────────
if df_data is None or len(df_data) == 0:
    st.error("Data tidak tersedia. Menggunakan data demo.")
    df_data = buat_data_demo()
    nama_dataset = "Data Demo (fallback)"
    interval_data = "bulanan"

st.info(f"📂 Sumber data aktif: **{nama_dataset}** — {len(df_data)} baris data")

# ── Proses: buat lag features & split data ───────────────────
df_lag = buat_lag_features(df_data, "nilai", n_lag)
fitur_cols = [f"lag_{i}" for i in range(1, n_lag + 1)]

# Pastikan cukup data setelah pembuatan lag
if len(df_lag) < 5:
    st.error(
        f"❌ Data terlalu sedikit setelah pembuatan {n_lag} lag features. "
        "Kurangi jumlah lag atau gunakan data yang lebih panjang."
    )
    st.stop()

# Train/test split 80/20 (time-aware, no shuffle)
split_idx = int(len(df_lag) * 0.8)
if split_idx < 2:
    split_idx = 2

df_train = df_lag.iloc[:split_idx]
df_test = df_lag.iloc[split_idx:]

X_train = df_train[fitur_cols].values
y_train = df_train["nilai"].values
X_test = df_test[fitur_cols].values
y_test = df_test["nilai"].values

# Training model
model = pilih_model(model_dipilih)
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)

# ── Forecast iteratif ke depan ───────────────────────────────
# Buffer: n_lag nilai terakhir dari seluruh data historis
buffer = list(df_data["nilai"].values[-n_lag:])
forecast_values = []

for _ in range(n_forecast):
    fitur_next = np.array(buffer[-n_lag:][::-1]).reshape(1, -1)
    pred = model.predict(fitur_next)[0]
    forecast_values.append(pred)
    buffer.append(pred)

# Buat label tanggal forecast (lanjut dari tanggal terakhir)
tgl_terakhir = pd.Timestamp(df_data["tanggal"].values[-1])

def buat_tanggal_forecast(tgl_terakhir, n, interval):
    """Generate n tanggal berikutnya sesuai interval data."""
    hasil = []
    tgl = tgl_terakhir
    for _ in range(n):
        if "harian" in interval:
            tgl = tgl + pd.tseries.offsets.BDay(1)
        elif "mingguan" in interval:
            tgl = tgl + pd.DateOffset(weeks=1)
        else:
            tgl = tgl + pd.DateOffset(months=1)
        hasil.append(tgl)
    return hasil

tanggal_forecast = buat_tanggal_forecast(tgl_terakhir, n_forecast, interval_data)

df_forecast = pd.DataFrame({
    "tanggal": tanggal_forecast,
    "nilai_prediksi": np.round(forecast_values, 2)
})

# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["📈 Data & Prediksi", "📊 Evaluasi Model", "🔍 Data Mentah"])

# ── Tab 1: Data & Prediksi ────────────────────────────────────
with tab1:
    # Siapkan data untuk chart
    tgl_train = df_train["tanggal"].values
    tgl_test = df_test["tanggal"].values

    fig = go.Figure()

    # Garis biru solid: data train
    fig.add_trace(go.Scatter(
        x=tgl_train,
        y=df_train["nilai"].values,
        mode="lines",
        name="Train (Historis)",
        line=dict(color="royalblue", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Nilai: %{y:.2f}<extra>Train</extra>"
    ))

    # Garis hijau solid: data aktual test
    fig.add_trace(go.Scatter(
        x=tgl_test,
        y=y_test,
        mode="lines",
        name="Test (Aktual)",
        line=dict(color="seagreen", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Nilai: %{y:.2f}<extra>Test Aktual</extra>"
    ))

    # Garis merah putus-putus: prediksi test
    fig.add_trace(go.Scatter(
        x=tgl_test,
        y=y_pred_test,
        mode="lines",
        name="Prediksi (Test)",
        line=dict(color="crimson", width=2, dash="dash"),
        hovertemplate="%{x|%Y-%m-%d}<br>Prediksi: %{y:.2f}<extra>Prediksi Test</extra>"
    ))

    # Garis oranye putus-putus: forecast ke depan
    fig.add_trace(go.Scatter(
        x=df_forecast["tanggal"],
        y=df_forecast["nilai_prediksi"],
        mode="lines+markers",
        name="Forecast",
        line=dict(color="darkorange", width=2, dash="dot"),
        marker=dict(size=6),
        hovertemplate="%{x|%Y-%m-%d}<br>Forecast: %{y:.2f}<extra>Forecast</extra>"
    ))

    # Shaded area: batas train/test
    if len(tgl_test) > 0:
        x_split_train_test = str(pd.Timestamp(tgl_test[0]))
        fig.add_vrect(
            x0=x_split_train_test, x1=x_split_train_test,
            line_width=1.5, line_color="gray", line_dash="dot",
            annotation_text="Train | Test",
            annotation_position="top left",
            annotation_font_color="gray"
        )

    # Shaded area: batas test/forecast
    if len(tanggal_forecast) > 0:
        x_split_test_fc = str(tanggal_forecast[0])
        fig.add_vrect(
            x0=x_split_test_fc, x1=x_split_test_fc,
            line_width=1.5, line_color="orange", line_dash="dot",
            annotation_text="Test | Forecast",
            annotation_position="top left",
            annotation_font_color="darkorange"
        )

    fig.update_layout(
        title=f"Prediksi {model_dipilih} — {nama_dataset}",
        xaxis_title="Tanggal",
        yaxis_title="Nilai",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
        height=480
    )
    st.plotly_chart(fig, use_container_width=True)

    # Tabel forecast
    st.subheader("📋 Tabel Forecast ke Depan")
    df_forecast_display = df_forecast.copy()
    df_forecast_display.columns = ["Periode", "Nilai Prediksi"]
    df_forecast_display["Periode"] = df_forecast_display["Periode"].dt.strftime("%Y-%m-%d")
    df_forecast_display["Nilai Prediksi"] = df_forecast_display["Nilai Prediksi"].map("{:.2f}".format)
    st.dataframe(df_forecast_display, use_container_width=True)

# ── Tab 2: Evaluasi Model ─────────────────────────────────────
with tab2:
    mae, rmse, r2 = hitung_metrik(y_test, y_pred_test)
    mae_b, rmse_b, r2_b = hitung_baseline_naive(y_test, y_train[-1])

    col1, col2, col3 = st.columns(3)
    col1.metric(
        "MAE",
        f"{mae:.2f}",
        delta=f"{mae_b - mae:.2f} vs Naive",
        delta_color="inverse"
    )
    col2.metric(
        "RMSE",
        f"{rmse:.2f}",
        delta=f"{rmse_b - rmse:.2f} vs Naive",
        delta_color="inverse"
    )
    col3.metric(
        "R²",
        f"{r2:.4f}",
        delta=f"{r2 - r2_b:.4f} vs Naive",
        delta_color="normal"
    )

    st.caption("Evaluasi dihitung pada test set (20% terakhir data historis)")

    # Scatter: Aktual vs Prediksi
    fig_scatter = go.Figure()

    fig_scatter.add_trace(go.Scatter(
        x=y_test,
        y=y_pred_test,
        mode="markers",
        name="Aktual vs Prediksi",
        marker=dict(color="royalblue", size=8, opacity=0.7),
        hovertemplate="Aktual: %{x:.2f}<br>Prediksi: %{y:.2f}<extra></extra>"
    ))

    # Garis diagonal perfect prediction
    min_val = float(min(y_test.min(), y_pred_test.min()))
    max_val = float(max(y_test.max(), y_pred_test.max()))
    fig_scatter.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        name="Perfect Prediction",
        line=dict(color="gray", dash="dash", width=1.5)
    ))

    fig_scatter.update_layout(
        title="Aktual vs Prediksi (Test Set)",
        xaxis_title="Nilai Aktual",
        yaxis_title="Nilai Prediksi",
        height=400
    )
    st.plotly_chart(fig_scatter, use_container_width=True)

# ── Tab 3: Data Mentah ────────────────────────────────────────
with tab3:
    st.subheader("Data Lengkap")
    df_display = df_data.copy()
    df_display["tanggal"] = pd.to_datetime(df_display["tanggal"]).dt.strftime("%Y-%m-%d")
    df_display["nilai"] = df_display["nilai"].map("{:.2f}".format)
    st.dataframe(df_display, use_container_width=True)

    csv_bytes = df_data.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="⬇️ Download CSV",
        data=csv_bytes,
        file_name=f"data_{nama_dataset.replace(' ', '_')}.csv",
        mime="text/csv"
    )

---
## 👥 CELL 4 — Tulis File `app_segmentasi.py`

In [ ]:
%%writefile app_segmentasi.py
# ============================================================
# APP 2: CUSTOMER SEGMENTATION DEMO
# Unsupervised Learning — KMeans Clustering
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="👥 Customer Segmentation Demo",
    page_icon="👥",
    layout="wide"
)

st.title("👥 Customer Segmentation")
st.markdown(
    "Demo interaktif segmentasi pelanggan menggunakan "
    "Unsupervised Learning (KMeans Clustering). "
    "Geser slider K dan eksplorasi bagaimana cluster terbentuk."
)

# ── Fungsi: buat data demo ───────────────────────────────────
def buat_data_demo():
    """Generate 200 pelanggan sintetis dengan 3 cluster, seed=42."""
    rng = np.random.RandomState(42)

    # Cluster A: Pelanggan Pasif
    n_a = 60
    usia_a = rng.uniform(20, 30, n_a)
    pengeluaran_a = rng.uniform(200, 400, n_a)
    frekuensi_a = rng.uniform(1, 3, n_a)

    # Cluster B: Pelanggan Potensial
    n_b = 80
    usia_b = rng.uniform(30, 50, n_b)
    pengeluaran_b = rng.uniform(500, 900, n_b)
    frekuensi_b = rng.uniform(4, 7, n_b)

    # Cluster C: Pelanggan Premium
    n_c = 60
    usia_c = rng.uniform(35, 60, n_c)
    pengeluaran_c = rng.uniform(1000, 2000, n_c)
    frekuensi_c = rng.uniform(8, 15, n_c)

    usia = np.concatenate([usia_a, usia_b, usia_c])
    pengeluaran = np.concatenate([pengeluaran_a, pengeluaran_b, pengeluaran_c])
    frekuensi = np.concatenate([frekuensi_a, frekuensi_b, frekuensi_c])

    # Tambah noise kecil agar tidak terlalu perfect
    usia += rng.normal(0, 1, len(usia))
    pengeluaran += rng.normal(0, 20, len(pengeluaran))
    frekuensi += rng.normal(0, 0.3, len(frekuensi))

    df = pd.DataFrame({
        "usia": np.round(usia, 1),
        "pengeluaran_bulanan": np.round(pengeluaran, 1),
        "frekuensi_transaksi": np.round(frekuensi, 1)
    })
    return df

# ── Fungsi: assign label interpretatif ──────────────────────
def assign_label_cluster(labels_array, df_asli, kolom_proxy, k, is_demo):
    """
    Ranking cluster berdasarkan rata-rata kolom_proxy.
    Tertinggi → Premium, Terendah → Pasif, Tengah → Potensial.
    """
    rata_per_cluster = {}
    for c in range(k):
        mask = labels_array == c
        if mask.sum() > 0:
            rata_per_cluster[c] = df_asli.loc[mask, kolom_proxy].mean()
        else:
            rata_per_cluster[c] = 0

    # Urutkan cluster dari pengeluaran tertinggi ke terendah
    sorted_clusters = sorted(rata_per_cluster.items(), key=lambda x: x[1], reverse=True)

    label_map = {}
    n_middle = k - 2  # jumlah cluster "Potensial"

    for rank, (cluster_id, _) in enumerate(sorted_clusters):
        if rank == 0:
            if is_demo:
                label_map[cluster_id] = "💎 Cluster Premium / Premium Customers"
            else:
                label_map[cluster_id] = "🔵 Cluster Tinggi / High"
        elif rank == len(sorted_clusters) - 1:
            if is_demo:
                label_map[cluster_id] = "😴 Cluster Pasif / Passive Customers"
            else:
                label_map[cluster_id] = "🔴 Cluster Rendah / Low"
        else:
            middle_rank = rank - 1  # index di antara Tinggi dan Rendah
            if n_middle == 1:
                if is_demo:
                    label_map[cluster_id] = "🌱 Cluster Potensial / Potential Customers"
                else:
                    label_map[cluster_id] = "🟡 Cluster Menengah / Medium"
            else:
                if is_demo:
                    label_map[cluster_id] = f"🌱 Cluster Potensial {middle_rank + 1}"
                else:
                    label_map[cluster_id] = f"🟡 Cluster Menengah {middle_rank + 1}"

    return label_map

# ── Fungsi: buat narasi cluster dinamis ─────────────────────
def buat_narasi_cluster(label, rata_rata_dict, is_demo):
    """Buat teks deskripsi dan rekomendasi berdasarkan label dan nilai rata-rata."""
    fitur_str = ", ".join([f"**{k}**: {v:.2f}" for k, v in rata_rata_dict.items()])

    if "Premium" in label or "Tinggi" in label:
        emoji = "💎"
        deskripsi = (
            f"Cluster ini memiliki profil tertinggi di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini menunjukkan keterlibatan dan nilai transaksi yang sangat tinggi."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Pertahankan dengan program loyalitas eksklusif, "
            "penawaran VIP, dan layanan personal. Jangan kehilangan segmen ini."
        )
    elif "Pasif" in label or "Rendah" in label:
        emoji = "😴"
        deskripsi = (
            f"Cluster ini memiliki profil terendah di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini jarang bertransaksi dan cenderung pasif."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Aktifkan kembali dengan diskon re-engagement, "
            "notifikasi promosi, atau kampanye win-back."
        )
    else:
        emoji = "🌱"
        deskripsi = (
            f"Cluster ini memiliki profil menengah. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini memiliki potensi untuk naik ke tier premium."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Dorong dengan program upsell dan reward bertahap. "
            "Target konversi ke tier premium melalui penawaran yang tepat sasaran."
        )

    return emoji, deskripsi, rekomendasi

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    sumber_data = st.radio(
        "Pilih Sumber Data",
        options=["👥 Data Demo (Pelanggan Sintetis)", "📁 Upload CSV Sendiri"]
    )

    df_data = None
    fitur_cols = None
    is_demo = True

    # ── Opsi 1: Data Demo ────────────────────────────────────
    if sumber_data == "👥 Data Demo (Pelanggan Sintetis)":
        df_data = buat_data_demo()
        fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
        is_demo = True

    # ── Opsi 2: Upload CSV ───────────────────────────────────
    else:
        is_demo = False
        st.markdown(
            "**Format:** Minimal 2 kolom numerik, maksimal 10 kolom. "
            "Kolom non-numerik diabaikan otomatis."
        )
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])

        if file_upload is not None:
            try:
                df_upload = pd.read_csv(file_upload)
                # Ambil hanya kolom numerik
                df_numerik = df_upload.select_dtypes(include=[np.number])
                if len(df_numerik.columns) < 2:
                    st.error("❌ File harus memiliki minimal 2 kolom numerik.")
                else:
                    kolom_numerik = list(df_numerik.columns[:10])
                    st.write("**Preview 5 baris pertama:**")
                    st.dataframe(df_upload.head())
                    fitur_terpilih = st.multiselect(
                        "Pilih kolom untuk clustering (min. 2)",
                        options=kolom_numerik,
                        default=kolom_numerik[:min(3, len(kolom_numerik))]
                    )
                    if len(fitur_terpilih) >= 2:
                        df_data = df_upload[fitur_terpilih].dropna().reset_index(drop=True)
                        fitur_cols = fitur_terpilih
                        st.success(f"✅ Data berhasil diproses: {len(df_data)} baris, {len(fitur_cols)} fitur")
                    else:
                        st.warning("⚠️ Pilih minimal 2 kolom untuk melanjutkan.")
            except Exception as e:
                st.error(f"❌ Gagal membaca file: {e}")

        # Fallback ke demo jika upload belum ada
        if df_data is None:
            df_data = buat_data_demo()
            fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
            is_demo = True

    st.divider()

    # Slider jumlah cluster
    k = st.slider("Jumlah Cluster (K)", min_value=2, max_value=6, value=3)

    # Checkbox interpretasi
    tampilkan_interpretasi = st.checkbox("💡 Tampilkan Interpretasi Cluster", value=False)

# ── Pastikan data tersedia ────────────────────────────────────
if df_data is None or fitur_cols is None:
    st.error("Data tidak tersedia.")
    st.stop()

# ── Proses: Scaling + KMeans ──────────────────────────────────
X = df_data[fitur_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)
df_data = df_data.copy()
df_data["cluster_id"] = labels

# ── Kolom proxy untuk interpretasi ───────────────────────────
if is_demo and "pengeluaran_bulanan" in fitur_cols:
    kolom_proxy = "pengeluaran_bulanan"
else:
    kolom_proxy = fitur_cols[0]

# ── Map label cluster ─────────────────────────────────────────
label_map = assign_label_cluster(labels, df_data, kolom_proxy, k, is_demo)

if tampilkan_interpretasi:
    df_data["cluster_label"] = df_data["cluster_id"].map(label_map)
else:
    df_data["cluster_label"] = df_data["cluster_id"].map(
        {c: f"Cluster {c}" for c in range(k)}
    )

# ── PCA 2D untuk visualisasi ──────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_data["pca_x"] = X_pca[:, 0]
df_data["pca_y"] = X_pca[:, 1]

# Centroid PCA
centroid_scaled = kmeans.cluster_centers_
centroid_pca = pca.transform(centroid_scaled)

# ── Warna cluster (color-blind friendly) ─────────────────────
WARNA_CLUSTER = [
    "#2196F3",  # biru
    "#FF9800",  # oranye
    "#4CAF50",  # hijau
    "#E91E63",  # merah muda
    "#9C27B0",  # ungu
    "#00BCD4",  # cyan
]

# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs([
    "🗺️ Visualisasi Cluster",
    "📊 Profil Cluster",
    "📉 Elbow Method"
])

# ── Tab 1: Visualisasi Cluster ────────────────────────────────
with tab1:
    fig = go.Figure()

    for idx_c in range(k):
        mask = df_data["cluster_id"] == idx_c
        subset = df_data[mask]
        label_c = subset["cluster_label"].iloc[0] if len(subset) > 0 else f"Cluster {idx_c}"
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]

        # Teks hover: semua fitur asli
        hover_teks = [
            "<br>".join([f"{col}: {row[col]:.2f}" for col in fitur_cols])
            + f"<br><b>{label_c}</b>"
            for _, row in subset.iterrows()
        ]

        fig.add_trace(go.Scatter(
            x=subset["pca_x"],
            y=subset["pca_y"],
            mode="markers",
            name=label_c,
            marker=dict(color=warna, size=8, opacity=0.75),
            text=hover_teks,
            hovertemplate="%{text}<extra></extra>"
        ))

    # Centroid sebagai bintang
    for idx_c in range(k):
        label_c = label_map.get(idx_c, f"Cluster {idx_c}")
        fig.add_trace(go.Scatter(
            x=[centroid_pca[idx_c, 0]],
            y=[centroid_pca[idx_c, 1]],
            mode="markers",
            name=f"Centroid {idx_c}",
            marker=dict(
                symbol="star",
                size=20,
                color="black",
                line=dict(color="white", width=1.5)
            ),
            showlegend=False,
            hovertemplate=f"Centroid {label_c}<extra></extra>"
        ))

    fig.update_layout(
        title=f"Visualisasi Cluster (K={k}) — PCA 2D",
        xaxis_title="PCA Komponen 1",
        yaxis_title="PCA Komponen 2",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=500
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption(
        "📌 PCA digunakan hanya untuk visualisasi 2D. "
        "Proses clustering dilakukan pada data asli setelah scaling."
    )

# ── Tab 2: Profil Cluster ─────────────────────────────────────
with tab2:
    # Tabel rata-rata per cluster
    df_profil = df_data.groupby("cluster_label")[fitur_cols].mean().round(2)
    st.subheader("Rata-rata Fitur per Cluster")
    st.dataframe(df_profil, use_container_width=True)

    # Grouped bar chart
    df_profil_reset = df_profil.reset_index()
    fig_bar = go.Figure()

    for idx_c, row in df_profil_reset.iterrows():
        label_c = row["cluster_label"]
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]
        fig_bar.add_trace(go.Bar(
            name=label_c,
            x=fitur_cols,
            y=[row[col] for col in fitur_cols],
            marker_color=warna
        ))

    fig_bar.update_layout(
        title="Perbandingan Rata-rata Fitur per Cluster",
        barmode="group",
        xaxis_title="Fitur",
        yaxis_title="Nilai Rata-rata",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=400
    )
    st.plotly_chart(fig_bar, use_container_width=True)

    # Narasi interpretatif jika diaktifkan
    if tampilkan_interpretasi:
        st.subheader("📝 Interpretasi Cluster")
        for idx_c in range(k):
            label_c = label_map.get(idx_c, f"Cluster {idx_c}")
            mask = df_data["cluster_id"] == idx_c
            rata_dict = {
                col: df_data.loc[mask, col].mean()
                for col in fitur_cols
            }
            emoji, deskripsi, rekomendasi = buat_narasi_cluster(label_c, rata_dict, is_demo)

            with st.expander(f"{emoji} {label_c} ({mask.sum()} pelanggan)"):
                st.markdown(deskripsi)
                st.markdown(rekomendasi)

# ── Tab 3: Elbow Method ───────────────────────────────────────
with tab3:
    # Hitung inertia untuk K=1 sampai 10
    k_range = list(range(1, 11))
    inertia_list = []
    for k_test in k_range:
        km_test = KMeans(n_clusters=k_test, n_init=10, random_state=42)
        km_test.fit(X_scaled)
        inertia_list.append(km_test.inertia_)

    fig_elbow = go.Figure()

    # Garis biru: inertia
    fig_elbow.add_trace(go.Scatter(
        x=k_range,
        y=inertia_list,
        mode="lines+markers",
        name="Inertia",
        line=dict(color="royalblue", width=2),
        marker=dict(size=7, color="royalblue"),
        hovertemplate="K=%{x}<br>Inertia=%{y:.2f}<extra></extra>"
    ))

    # Marker merah: K yang sedang dipilih
    idx_k = k - 1  # index di k_range
    fig_elbow.add_trace(go.Scatter(
        x=[k],
        y=[inertia_list[idx_k]],
        mode="markers",
        name=f"K={k} (dipilih)",
        marker=dict(size=15, color="crimson", symbol="circle"),
        hovertemplate=f"K={k}<br>Inertia={inertia_list[idx_k]:.2f}<extra>K dipilih</extra>"
    ))

    fig_elbow.update_layout(
        title="Elbow Method — Mencari K Optimal",
        xaxis_title="Jumlah Cluster (K)",
        yaxis_title="Inertia",
        xaxis=dict(tickmode="linear", dtick=1),
        height=420
    )
    st.plotly_chart(fig_elbow, use_container_width=True)

    st.info(
        "📖 Cara membaca grafik ini: Pilih nilai K di titik 'siku' (elbow), "
        "yaitu titik di mana penurunan inertia mulai melambat secara signifikan. "
        "Menambah K setelah titik ini tidak memberikan manfaat yang sebanding."
    )

---
## 🔮 CELL 5 — Jalankan App 1: Time Series Forecasting

> Tunggu link ngrok muncul, lalu klik untuk membuka di tab baru.

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

def run_app1():
    subprocess.Popen(
        ["streamlit", "run", "app_timeseries.py",
         "--server.port", "8501",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=run_app1, daemon=True).start()
time.sleep(5)

tunnel1 = ngrok.connect(8501)
print("=" * 55)
print("🔮 TIME SERIES FORECASTING APP — SIAP DIGUNAKAN")
print(f"🌐 Buka di browser: {tunnel1.public_url}")
print("=" * 55)

---
## 👥 CELL 6 — Jalankan App 2: Customer Segmentation

> Tunggu link ngrok muncul, lalu klik untuk membuka di tab baru.

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

def run_app2():
    subprocess.Popen(
        ["streamlit", "run", "app_segmentasi.py",
         "--server.port", "8502",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=run_app2, daemon=True).start()
time.sleep(5)

tunnel2 = ngrok.connect(8502)
print("=" * 55)
print("👥 CUSTOMER SEGMENTATION APP — SIAP DIGUNAKAN")
print(f"🌐 Buka di browser: {tunnel2.public_url}")
print("=" * 55)

---
## 🛑 CELL 7 — Hentikan Semua Tunnel Ngrok

> **Jangan jalankan cell ini saat Run All!**
>
> Cell ini sengaja dikomentari agar tidak ikut berjalan saat **Runtime → Run all**.
> Jalankan **hanya jika ingin menghentikan** semua tunnel ngrok secara manual.
>
> **Cara pakai:** Hapus tanda `#` pada baris kode di bawah, lalu klik ▶ jalankan cell ini.

In [ ]:
# Hapus tanda # di bawah ini, lalu jalankan cell ini untuk menghentikan semua tunnel

# ngrok.kill()
# print("✅ Semua tunnel ngrok telah dihentikan.")